# Savitzky-Golay step-edge example

A square-window polynomial gradient filter. This notebook creates a simple black-to-white step edge, runs the filter, and plots the result.


## How this filter works

Savitzky-Golay filtering fits a low-degree polynomial over a local window and uses the fitted polynomial to estimate a smoothed value or derivative. This notebook uses the derivative form, so `degree` controls the local polynomial and `radius` controls the window size.

### Paper reference

Use Savitzky and Golay's 1964 least-squares smoothing and differentiation paper, DOI [10.1021/ac60214a047](https://doi.org/10.1021/ac60214a047). Links. [Savitzky-Golay filter](https://en.wikipedia.org/wiki/Savitzky%E2%80%93Golay_filter).


## Setup

Import the small set of tools used in this notebook. The autoreload lines help Jupyter pick up local source edits without restarting the kernel every time.


In [ ]:
# Imports and local reloads.
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import torch

from agfb_filters import ExecutionPath, get_filter_definition, run_filter

## Filter settings

Choose the filter settings here. The `definition` describes the filter, and the `path` chooses the matching execution path.


In [ ]:
# Change these settings to try the filter.
radius = 2
degree = 2
definition = get_filter_definition("savitzky_golay", radius=radius, degree=degree)
path = ExecutionPath.SPATIAL_DENSE

## Test image

Create a test image. By default this is a synthetic step edge. The commented block lets you replace it with your own image file path.


In [ ]:
# Build the step-edge test image.
size = 1024
image = torch.zeros(1, size, size)
image[:, :, size // 2 :] = 1.0

# To use your own image instead, uncomment this block and set image_path.
# image_path = "path/to/image.png"
# image = torch.as_tensor(plt.imread(image_path), dtype=torch.float32)
# if image.ndim == 3:
#     image = image[..., :3].mean(dim=-1)
# image = image.unsqueeze(0)

plt.figure(figsize=(4, 4))
plt.imshow(image[0], cmap="gray", vmin=0, vmax=1)
plt.title("input image")
plt.axis("off")

## Filter weights

Visualize the fixed filter weights. `kernel_x` responds to left-to-right changes, and `kernel_y` responds to top-to-bottom changes.


In [ ]:
# Plot the filter weights.
kernel_x, kernel_y = definition.dense_kernels()
weight_limit = max(float(kernel_x.abs().max()), float(kernel_y.abs().max()))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

for ax, kernel, title in zip(
    axes,
    (kernel_x, kernel_y),
    ("kernel_x", "kernel_y"),
    strict=False,
):
    ax.imshow(kernel, cmap="gray", vmin=-weight_limit, vmax=weight_limit)
    ax.set_title(title)
    ax.axis("off")

fig.suptitle(f"{definition.name} weights")
fig.tight_layout()

## Run the filter

Run the filter. The timer measures only the filter call, not the image setup or plotting.


In [ ]:
# Time the filter call.
start = time.perf_counter()
gradient_x, gradient_y = run_filter(
    definition,
    image,
    path=path,
    boundary=definition.default_boundary,
)
elapsed = time.perf_counter() - start

print(f"{elapsed:.4f} seconds")

## Gradient results

Plot the two gradient outputs with the same grayscale range. The input changes left-to-right, so the strongest response should usually be in `gradient_x`.


In [ ]:
# Plot gradient_x and gradient_y.
limit = max(float(gradient_x.abs().max()), float(gradient_y.abs().max()))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

axes[0].imshow(gradient_x[0], cmap="gray", vmin=-limit, vmax=limit)
axes[0].set_title("gradient_x")

axes[1].imshow(gradient_y[0], cmap="gray", vmin=-limit, vmax=limit)
axes[1].set_title("gradient_y")

for ax in axes:
    ax.axis("off")

fig.suptitle(f"{definition.name}, time={elapsed:.4f} s")
fig.tight_layout()